<a href="https://colab.research.google.com/github/JouichatKhadija/.github.io/blob/main/TP1_PFA_RAG_4eme_Ann%C3%A9e_IIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: scikit-learn
    Found ex

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

print("Bibliothèques importées")

print("\n DOCUMENT")

text = """
Blockchain functionality
We have now defined and described blockchain. Now, let's see how a blockchain works. Nodes are either miners that create new blocks and mint cryptocurrency (coins) or block signers that validate and digitally sign the transactions. A critical decision that every blockchain network must make is to figure out which node will append the next block to the blockchain. This decision is made using a consensus mechanism.

Consensus is a process of achieving agreement between distrusting nodes on the final state of data. To achieve consensus, different algorithms are used. It is easy to reach an agreement in a centralized network (in client-server systems, for example), but when multiple nodes are participating in a distributed system and they need to agree on a single value, it becomes quite a challenge to achieve consensus. This process of attaining agreement on a common state or value among multiple nodes is known as distributed consensus. If faults are allowed, then we call such a mechanism fault tolerant distributed consensus, where despite the failure of some nodes, agreement is reached between them.

Now, we will look at how a blockchain validates transactions and creates and adds blocks to grow the blockchain, using a general scheme for creating blocks:

Transaction is initiated: A node starts a transaction by first creating it and then digitally signing it with its private key. A transaction can represent various actions in a blockchain. Most commonly, this is a data structure that represents the transfer of value between users on the blockchain network. The transaction data structure usually consists of some logic of transfer of value, relevant rules, source and destination addresses, and other validation information.

Transactions are usually either a cryptocurrency transfer (transfer of value) or a smart contract invocation that can perform any desired operation. A transaction occurs between two or more parties. This will be covered in more detail in specific chapters on Bitcoin and Ethereum later in the book.

Transaction is validated and broadcast: A transaction is propagated (broadcast) usually by using data-dissemination protocols, such as the Gossip protocol, to other peers that validate the transaction based on preset validity criteria. Before a transaction is propagated, it is also verified to ensure that it is valid.

Find new block: When the transaction is received and validated by special participants called miners on the blockchain network, it is included in a block, and the process of mining starts. This process is also sometimes referred to as "finding a new block." Here, nodes called miners race to finalize the block they've created by a process known as mining.

New block found: Once a miner solves a mathematical puzzle (or fulfills the requirements of the consensus mechanism implemented in a blockchain), the block is considered "found" and finalized. At this point, the transaction is considered confirmed. Usually, in cryptocurrency blockchains such as Bitcoin, the miner who solves the mathematical puzzle is also rewarded with a certain number of coins as an incentive for their effort and the resources they spent in the mining process.

Add new block to the blockchain: The newly created block is validated, transactions or smart contracts within it are executed, and it is propagated to other peers. Peers also validate and execute the block. It now becomes part of the blockchain (ledger), and the next block links itself cryptographically back to this block. This link is called a hash pointer.
"""

print("Document chargé")
print(f"Longueur du texte: {len(text)} caractères")
print(f"Extrait du début:\n{text[:300]}...")



Bibliothèques importées

 DOCUMENT
Document chargé
Longueur du texte: 3600 caractères
Extrait du début:

Blockchain functionality
We have now defined and described blockchain. Now, let's see how a blockchain works. Nodes are either miners that create new blocks and mint cryptocurrency (coins) or block signers that validate and digitally sign the transactions. A critical decision that every blockchain ...


In [3]:
print("\n CHUNKS :")

splitter = CharacterTextSplitter(
chunk_size=100,
chunk_overlap=20
)


 CHUNKS :


In [4]:
docs = splitter.split_text(text)
print(docs)

["Blockchain functionality\nWe have now defined and described blockchain. Now, let's see how a blockchain works. Nodes are either miners that create new blocks and mint cryptocurrency (coins) or block signers that validate and digitally sign the transactions. A critical decision that every blockchain network must make is to figure out which node will append the next block to the blockchain. This decision is made using a consensus mechanism.", 'Consensus is a process of achieving agreement between distrusting nodes on the final state of data. To achieve consensus, different algorithms are used. It is easy to reach an agreement in a centralized network (in client-server systems, for example), but when multiple nodes are participating in a distributed system and they need to agree on a single value, it becomes quite a challenge to achieve consensus. This process of attaining agreement on a common state or value among multiple nodes is known as distributed consensus. If faults are allowed,

In [5]:
print("\n EMBEDDINGS:")

embeddings = HuggingFaceEmbeddings(
model_name="sentence-transformers/all-MiniLM-L6-v2"
)



 EMBEDDINGS:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
vectorstore = FAISS.from_texts(docs, embeddings)

In [11]:
query = "How does consensus mechanism work in blockchain?"

In [12]:
results = vectorstore.similarity_search(query, k=2)

In [13]:
for r in results:
 print(r.page_content)

Blockchain functionality
We have now defined and described blockchain. Now, let's see how a blockchain works. Nodes are either miners that create new blocks and mint cryptocurrency (coins) or block signers that validate and digitally sign the transactions. A critical decision that every blockchain network must make is to figure out which node will append the next block to the blockchain. This decision is made using a consensus mechanism.
Consensus is a process of achieving agreement between distrusting nodes on the final state of data. To achieve consensus, different algorithms are used. It is easy to reach an agreement in a centralized network (in client-server systems, for example), but when multiple nodes are participating in a distributed system and they need to agree on a single value, it becomes quite a challenge to achieve consensus. This process of attaining agreement on a common state or value among multiple nodes is known as distributed consensus. If faults are allowed, then 